Скачиваем репозиторий:

In [ ]:
!git clone --branch asr https://github.com/gravitsapa/DLA-HW1.git

Переходим в папку репозитория:

In [ ]:
%cd DLA-HW1

Ставим зависимости:

In [ ]:
%pip install -q -r requirements.txt

Скачиваем веса модели и логи инференса:

In [ ]:
!python3 download_resources.py

Результаты финальной модели conformer + rnn-t:

In [ ]:
!cat saved/inference/info.log

Скачиваем кастомный датасет. Введите ссылку на него в input-поле, которое появится под ячейкой:

In [ ]:
import zipfile
import gdown
from pathlib import Path

# ссылка должна быть типа https://drive.google.com/uc?id=1Ofb7cw4vfbOffA03-A8hSNpG-dxcQdb2
url = input("dataset archive URL: ").strip()
custom = Path("data/custom_archive")
Path("data/custom_archive").mkdir(exist_ok=True, parents=True)
assert gdown.download(url, "data/custom_archive/archive.zip")

with zipfile.ZipFile("data/custom_archive/archive.zip") as source:
    source.extractall(custom)
dataset = next(path.parent for path in custom.rglob("audio") if path.is_dir())


Запускаем инференс на кастомном датасете

In [ ]:
!python3 inference.py --config-name inference_custom dataloader.batch_size=32 custom_dataset_dir="{dataset}" experiment_name=custom_exp inferencer.from_pretrained=saved/model.pth

Считаем метрики предсказанного результата:

In [ ]:
!python3 calc_metrics.py --dataset-dir "{dataset}" --predictions-dir saved/inference/custom_exp/predictions/custom/transcriptions

Если дополонительно хотим запустить инференс модели на librispeech test-other и test-clean:

In [ ]:
# !python3 inference.py